In [4]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.visualization import plot_histogram
from qiskit.result import marginal_counts
from qiskit.circuit import ControlledGate
from qiskit.circuit.classical import expr
import math

In [7]:
from qiskit import QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import numpy as np


def quantum_secret_sharing(theta=np.pi / 3, phi=np.pi / 4):
    """
    Implements the quantum secret sharing protocol.

    Alice's qubit x is in state: cos(theta/2)|0> + e^(i*phi)*sin(theta/2)|1>

    After the protocol, Carol's qubit c should be in the same state.

    Qubits:  x=0, a=1, b=2, c=3
    """
    # ── Registers ──────────────────────────────────────────────────────────────
    q = QuantumRegister(4, "q")          # x, a, b, c
    c_bell = ClassicalRegister(2, "bell")  # r_x, r_a  (Alice's Bell measurement)
    c_bob = ClassicalRegister(1, "bob")    # r_b        (Bob's diagonal measurement)
    c_carol = ClassicalRegister(1, "carol")  # final readout of Carol's qubit

    qc = QuantumCircuit(q, c_bell, c_bob, c_carol)

    # ── Step 1: Prepare Alice's secret qubit x ─────────────────────────────────
    # x = cos(theta/2)|0> + e^(i*phi)*sin(theta/2)|1>
    qc.ry(theta, q[0])
    qc.p(phi, q[0])

    qc.barrier(label="x prepared")

    # ── Step 2: Prepare the GHZ state on a, b, c ──────────────────────────────
    # |GHZ> = 1/sqrt(2)(|000> + |111>)
    qc.h(q[1])          # H on a
    qc.cx(q[1], q[2])   # CNOT(a, b)
    qc.cx(q[1], q[3])   # CNOT(a, c)

    qc.barrier(label="GHZ ready")

    # ── Step 3: Alice entangles x with the GHZ state ──────────────────────────
    # CNOT(x, a), CNOT(x, b), CNOT(x, c)
    qc.cx(q[0], q[1])
    qc.cx(q[0], q[2])
    qc.cx(q[0], q[3])

    qc.barrier(label="x entangled with GHZ")

    # ── Step 4: Bell-basis measurement on x and a ──────────────────────────────
    # Apply cx(a, x) then h(a)  →  i.e. cx(1,0) then h(1) in 0-indexed
    qc.cx(q[1], q[0])
    qc.h(q[1])
    qc.measure(q[0], c_bell[0])   # r_x
    qc.measure(q[1], c_bell[1])   # r_a

    qc.barrier(label="Alice measures")

    # ── Step 5: Bob measures b in the diagonal (Hadamard) basis ────────────────
    qc.h(q[2])
    qc.measure(q[2], c_bob[0])    # r_b

    qc.barrier(label="Bob measures")

    # ── Step 6: Carol applies corrections to c ─────────────────────────────────
    # Truth table:
    #  r_x  r_a  r_b   operations
    #   0    0    0    none
    #   0    0    1    Z
    #   0    1    0    Z
    #   0    1    1    none
    #   1    0    0    X
    #   1    0    1    Z then X
    #   1    1    0    Z then X
    #   1    1    1    X
    #
    # Pattern: Z is applied when r_a XOR r_b == 1  (i.e. r_a != r_b)
    #          X is applied when r_x == 1
    #
    # Implemented with classically-controlled gates.

    # Z correction: apply Z when r_a XOR r_b == 1.
    # c_bell[1] = r_a,  c_bob[0] = r_b.
    # Two separate conditional Z's cancel when both bits are 1 (Z²=I),
    # so the net effect is: Z applied iff exactly one of r_a, r_b is 1. ✓
    with qc.if_test((c_bell[1], 1)):
        qc.z(q[3])
    with qc.if_test((c_bob[0], 1)):
        qc.z(q[3])

    # X correction: apply X when r_x == 1
    with qc.if_test((c_bell[0], 1)):
        qc.x(q[3])

    # ── Step 7: Measure Carol's qubit ─────────────────────────────────────────
    qc.measure(q[3], c_carol[0])

    return qc


def run_protocol(theta=np.pi / 3, phi=np.pi / 4, shots=4096):
    """
    Run the secret sharing protocol and verify the output.
    The expected output distribution of Carol's qubit should match
    what a direct measurement of |x> = cos(θ/2)|0> + e^(iφ)sin(θ/2)|1> gives.
    """
    qc = quantum_secret_sharing(theta, phi)

    simulator = AerSimulator()
    job = simulator.run(qc, shots=shots)
    result = job.result()
    counts = result.get_counts()

    # The measurement result string is "carol bob bell" (Qiskit reverses register order)
    # Extract just Carol's bit (leftmost in the output string)
    carol_counts = {"0": 0, "1": 0}
    for bitstring, count in counts.items():
        # bitstring format: "c_carol c_bob c_bell[1] c_bell[0]"  (Qiskit big-endian)
        carol_bit = bitstring[0]  # leftmost = highest register = c_carol
        carol_counts[carol_bit] = carol_counts.get(carol_bit, 0) + count

    prob_0 = carol_counts.get("0", 0) / shots
    prob_1 = carol_counts.get("1", 0) / shots

    # Theoretical expectation
    expected_prob_0 = np.cos(theta / 2) ** 2
    expected_prob_1 = np.sin(theta / 2) ** 2

    return {
        "carol_counts": carol_counts,
        "prob_0": prob_0,
        "prob_1": prob_1,
        "expected_prob_0": expected_prob_0,
        "expected_prob_1": expected_prob_1,
        "all_counts": counts,
    }


def run_tests():
    """
    Test the protocol across several different input states.
    """
    test_cases = [
        ("  |0>  ", 0.0, 0.0),
        ("  |1>  ", np.pi, 0.0),
        ("  |+>  ", np.pi / 2, 0.0),
        ("  |->  ", np.pi / 2, np.pi),
        (" 60°/45°", np.pi / 3, np.pi / 4),
        (" 90°/90°", np.pi / 2, np.pi / 2),
    ]

    print("=" * 65)
    print("  Quantum Secret Sharing – HBB98 Protocol Test")
    print("=" * 65)
    print(f"  {'State':<10} {'P(0) got':>10} {'P(0) exp':>10}  {'P(1) got':>10} {'P(1) exp':>10}  {'OK?':>5}")
    print("-" * 65)

    all_passed = True
    for label, theta, phi in test_cases:
        res = run_protocol(theta, phi, shots=8192)
        tol = 0.04  # allow ~4% statistical fluctuation
        ok_0 = abs(res["prob_0"] - res["expected_prob_0"]) < tol
        ok_1 = abs(res["prob_1"] - res["expected_prob_1"]) < tol
        ok = ok_0 and ok_1
        all_passed = all_passed and ok
        mark = "✓" if ok else "✗"
        print(
            f"  {label:<10} {res['prob_0']:>10.3f} {res['expected_prob_0']:>10.3f}"
            f"  {res['prob_1']:>10.3f} {res['expected_prob_1']:>10.3f}  {mark:>5}"
        )

    print("=" * 65)
    print(f"  All tests passed: {all_passed}")
    print("=" * 65)
    return all_passed


# ── Print the circuit for inspection ──────────────────────────────────────────
if __name__ == "__main__":
    print("\nCircuit diagram (x=q[0], a=q[1], b=q[2], c=q[3]):")
    print(quantum_secret_sharing().draw(output="text", fold=120))
    print()
    run_tests()


Circuit diagram (x=q[0], a=q[1], b=q[2], c=q[3]):
         ┌─────────┐┌────────┐ x prepared                 GHZ ready                 x entangled with GHZ ┌───┐     ┌─┐»
    q_0: ┤ Ry(π/3) ├┤ P(π/4) ├─────░──────────────────────────░───────■────■────■────────────░───────────┤ X ├─────┤M├»
         └─────────┘└────────┘     ░      ┌───┐               ░     ┌─┴─┐  │    │            ░           └─┬─┘┌───┐└╥┘»
    q_1: ──────────────────────────░──────┤ H ├──■────■───────░─────┤ X ├──┼────┼────────────░─────────────■──┤ H ├─╫─»
                                   ░      └───┘┌─┴─┐  │       ░     └───┘┌─┴─┐  │            ░                └───┘ ║ »
    q_2: ──────────────────────────░───────────┤ X ├──┼───────░──────────┤ X ├──┼────────────░──────────────────────╫─»
                                   ░           └───┘┌─┴─┐     ░          └───┘┌─┴─┐          ░                      ║ »
    q_3: ──────────────────────────░────────────────┤ X ├─────░───────────────┤ X ├──────────░───────────────